In [1]:
!pip install matplotlib pandas seaborn numpy nltk

In [2]:
%matplotlib inline

import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import _pickle as cPickle
import nltk

In [3]:
df = pd.read_csv('lsapp.tsv', sep='\t', skiprows=1, header=None, names=['user_id', 'session_id', 'timestamp', 'app_name', 'event_type'])
df_open = df.loc[df['event_type'] == 'Opened']

In [4]:
df

,user_id,session_id,timestamp,app_name,event_type
0,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened
1,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Closed
2,0,1,2018-01-16 06:01:07,Minesweeper Classic (Mines),Opened
3,0,1,2018-01-16 06:01:07,Minesweeper Classic (Mines),Closed
4,0,1,2018-01-16 06:01:08,Minesweeper Classic (Mines),Opened
...,...,...,...,...,...
3658584,291,76247,2018-04-06 14:35:15,Facebook,Closed
3658585,291,76247,2018-04-06 14:35:15,Facebook,Opened
3658586,291,76247,2018-04-06 14:35:37,Facebook,Closed
3658587,291,76247,2018-04-06 14:35:37,Facebook,Opened


In [5]:
df_open.loc[:,'timestamp'] = pd.to_datetime(df_open['timestamp'])
df.loc[:,'timestamp'] = pd.to_datetime(df['timestamp'])

In [6]:
df = df.iloc[:-1, :] # last row includes only NaNs and NaT (not a number)
df

,user_id,session_id,timestamp,app_name,event_type
0,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened
1,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Closed
2,0,1,2018-01-16 06:01:07,Minesweeper Classic (Mines),Opened
3,0,1,2018-01-16 06:01:07,Minesweeper Classic (Mines),Closed
4,0,1,2018-01-16 06:01:08,Minesweeper Classic (Mines),Opened
...,...,...,...,...,...
3658583,291,76247,2018-04-06 14:34:34,Facebook,Opened
3658584,291,76247,2018-04-06 14:35:15,Facebook,Closed
3658585,291,76247,2018-04-06 14:35:15,Facebook,Opened
3658586,291,76247,2018-04-06 14:35:37,Facebook,Closed


In [7]:
df['time_dff']  = df[['timestamp']].diff()
df['interaction_id'] = (((df.timestamp-df.timestamp.shift(1) > pd.Timedelta(1, 'm')) 
                              & (df.event_type == 'Opened')) 
                              | ~(df.app_name == df.app_name.shift(1))
                              | ~(df.user_id == df.user_id.shift(1))).cumsum()
df['session_id'] = (((df.timestamp-df.timestamp.shift(1) > pd.Timedelta(5, 'm')) 
                              & (df.event_type == 'Opened')) 
                              | ~(df.user_id == df.user_id.shift(1))).cumsum()
df_start = df.drop_duplicates(subset=['interaction_id'], keep='first')
df_end = df.drop_duplicates(subset=['interaction_id'], keep='last')
df_start.set_index('interaction_id', inplace=True)
df_end.set_index('interaction_id', inplace=True)
df_start.loc[:, 'open_time'] = df_start['timestamp']
df_start.loc[:, 'close_time']  = df_end['timestamp']

C:\Users\Brin\AppData\Local\Temp\ipykernel_17340\4188201099.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['time_dff']  = df[['timestamp']].diff()
C:\Users\Brin\AppData\Local\Temp\ipykernel_17340\4188201099.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['interaction_id'] = (((df.timestamp-df.timestamp.shift(1) > pd.Timedelta(1, 'm'))
C:\Users\Brin\AppData\Local\Temp\ipykernel_17340\4188201099.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Tr

In [15]:
df_start.head(20)

,user_id,session_id,timestamp,app_name,event_type,time_dff,open_time,close_time
interaction_id,,,,,,,,
1,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened,NaN,2018-01-16 06:01:05,2018-01-16 06:01:09
2,0,1,2018-01-16 06:03:44,Minesweeper Classic (Mines),Opened,0 days 00:02:35,2018-01-16 06:03:44,2018-01-16 06:04:17
3,0,1,2018-01-16 06:25:54,Gmail,User Interaction,0 days 00:21:37,2018-01-16 06:25:54,2018-01-16 06:25:54
4,0,1,2018-01-16 06:26:05,Google,Opened,0 days 00:00:11,2018-01-16 06:26:05,2018-01-16 06:26:10
5,0,1,2018-01-16 06:26:10,Instagram,Opened,0 days 00:00:00,2018-01-16 06:26:10,2018-01-16 06:26:21
6,0,1,2018-01-16 06:26:21,Google,Opened,0 days 00:00:00,2018-01-16 06:26:21,2018-01-16 06:26:26
7,0,1,2018-01-16 06:26:26,Google Chrome,Opened,0 days 00:00:00,2018-01-16 06:26:26,2018-01-16 06:36:26
8,0,2,2018-01-16 07:15:56,Minesweeper Classic (Mines),Opened,0 days 00:39:30,2018-01-16 07:15:56,2018-01-16 07:15:58
9,0,2,2018-01-16 07:20:45,Google Chrome,Opened,0 days 00:04:47,2018-01-16 07:20:45,2018-01-16 07:20:45


## Session time

### Sum of interaction times

In [9]:
# session time is sum of all interactions
df_start.groupby('session_id')['time_dff'].sum()

session_id
1        0 days 00:24:23
2        0 days 00:44:19
3        0 days 00:41:51
4        0 days 00:27:41
5        0 days 01:25:01
              ...       
33216    0 days 00:38:56
33217    0 days 03:50:05
33218    0 days 00:17:11
33219    0 days 00:26:59
33220    0 days 01:14:18
Name: time_dff, Length: 33220, dtype: object

### Time difference between first and last session interaction

In [18]:
# session time as difference between first session usage open_time and last session usage close_time
df_session_time = df_start.groupby('session_id').apply(lambda x: x['close_time'].max() - x['open_time'].min())
df_session_time.head(50)

session_id
1    0 days 00:35:21
2    0 days 00:05:48
3    0 days 00:02:06
4    0 days 00:03:30
5    0 days 00:44:03
6    0 days 00:01:59
7    0 days 00:00:32
8    0 days 00:00:15
9    0 days 00:07:52
10   0 days 00:00:52
11   0 days 00:02:09
12   0 days 00:08:40
13   0 days 00:00:28
14   0 days 00:43:05
15   0 days 00:00:30
16   0 days 00:04:49
17   0 days 00:03:15
18   0 days 00:01:49
19   0 days 00:01:10
20   0 days 00:05:46
21   0 days 00:03:47
22   0 days 00:00:45
23   0 days 01:17:13
24   0 days 00:00:07
25   0 days 00:00:02
26   0 days 00:00:00
27   0 days 00:00:26
28   0 days 00:00:00
29   0 days 00:00:01
30   0 days 00:00:00
31   1 days 14:49:19
32   0 days 00:00:54
33   0 days 01:44:17
34   0 days 00:00:09
35   0 days 00:00:01
36   0 days 01:33:08
37   0 days 00:00:02
38   0 days 00:00:18
39   0 days 00:15:54
40   0 days 00:00:22
41   0 days 00:02:10
42   0 days 00:02:36
43   0 days 00:03:00
44   0 days 00:11:16
45   0 days 01:07:23
46   0 days 00:22:45
47   0 days 00:08:13
48

In [16]:
# Compute mean and standard deviation of session times
mean_session_time = df_session_time.mean()
std_session_time = df_session_time.std()

mean_session_time, std_session_time

(Timedelta('0 days 01:21:17.672877784'),
 Timedelta('0 days 16:01:32.896302107'))

## Prediction

### Setup

In [25]:
# print number of users by counting unique user ids
df_start['user_id'].nunique()

292

In [21]:
# for each user create its own dataframe
df_user = df_start.groupby('user_id')

In [24]:
# print 10 rows of first user
df_user.get_group(0).head(10)

,user_id,session_id,timestamp,app_name,event_type,time_dff,open_time,close_time
interaction_id,,,,,,,,
1,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened,NaN,2018-01-16 06:01:05,2018-01-16 06:01:09
2,0,1,2018-01-16 06:03:44,Minesweeper Classic (Mines),Opened,0 days 00:02:35,2018-01-16 06:03:44,2018-01-16 06:04:17
3,0,1,2018-01-16 06:25:54,Gmail,User Interaction,0 days 00:21:37,2018-01-16 06:25:54,2018-01-16 06:25:54
4,0,1,2018-01-16 06:26:05,Google,Opened,0 days 00:00:11,2018-01-16 06:26:05,2018-01-16 06:26:10
5,0,1,2018-01-16 06:26:10,Instagram,Opened,0 days 00:00:00,2018-01-16 06:26:10,2018-01-16 06:26:21
6,0,1,2018-01-16 06:26:21,Google,Opened,0 days 00:00:00,2018-01-16 06:26:21,2018-01-16 06:26:26
7,0,1,2018-01-16 06:26:26,Google Chrome,Opened,0 days 00:00:00,2018-01-16 06:26:26,2018-01-16 06:36:26
8,0,2,2018-01-16 07:15:56,Minesweeper Classic (Mines),Opened,0 days 00:39:30,2018-01-16 07:15:56,2018-01-16 07:15:58
9,0,2,2018-01-16 07:20:45,Google Chrome,Opened,0 days 00:04:47,2018-01-16 07:20:45,2018-01-16 07:20:45


In [27]:
# for user 0 print all unique app names
df_user.get_group(0)['app_name'].unique()

array(['Minesweeper Classic (Mines)', 'Gmail', 'Google', 'Instagram',
       'Google Chrome', 'Clock', 'Maps', 'YouTube', 'Facebook',
       'Messages', 'Phone', 'Snapchat', 'Settings', 'Google Photos',
       'Hangouts', 'Amazon Shopping', 'Facebook Messenger',
       'Google Play Store', 'Calendar'], dtype=object)

### Random app

In [38]:
# create user to app dictionary including all unique app names for each user
user_to_app_dict = {}

for user_id, user_df in df_user:
    user_to_app_dict[user_id] = user_df['app_name'].unique()

# print user to app dictionary
user_to_app_dict


{0: array(['Minesweeper Classic (Mines)', 'Gmail', 'Google', 'Instagram',
        'Google Chrome', 'Clock', 'Maps', 'YouTube', 'Facebook',
        'Messages', 'Phone', 'Snapchat', 'Settings', 'Google Photos',
        'Hangouts', 'Amazon Shopping', 'Facebook Messenger',
        'Google Play Store', 'Calendar'], dtype=object),
 1: array(['Google Photos', 'Discord', 'Messages', 'Google', 'Snapchat',
        'Instagram', 'Google Chrome', 'Facebook Messenger', 'Facebook',
        'Google Drive', 'Twitter', 'Google Play Store', 'YouTube', 'Phone',
        'Spotify Music', 'Clock', 'Settings', 'Reddit'], dtype=object),
 2: array(['Android In Call UI', 'Receipt Hog', 'Gmail', 'Facebook',
        'Google Chrome', 'Instagram', 'Amazon Shopping', 'Google',
        'Snapchat', 'Maps', 'Ibotta', 'YouTube', 'Spotify Music',
        'Google Play Store', 'Settings', 'Facebook Messenger'],
       dtype=object),
 3: array(['PayPal Mobile Cash', 'Google Play Store', 'Google Chrome',
        'YouTube', 'A

In [40]:
# iterate through all df_start interactions and choose app_name as random app from user_to_app_dict for user_id
df_start['random_app_name'] = df_start.apply(lambda row: np.random.choice(user_to_app_dict[row['user_id']]), axis=1)
df_start.head(100)

C:\Users\Brin\AppData\Local\Temp\ipykernel_17340\1922601193.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_start['random_app_name'] = df_start.apply(lambda row: np.random.choice(user_to_app_dict[row['user_id']]), axis=1)


,user_id,session_id,timestamp,app_name,event_type,time_dff,open_time,close_time,random_app_name
interaction_id,,,,,,,,,
1,0,1,2018-01-16 06:01:05,Minesweeper Classic (Mines),Opened,NaN,2018-01-16 06:01:05,2018-01-16 06:01:09,Google Play Store
2,0,1,2018-01-16 06:03:44,Minesweeper Classic (Mines),Opened,0 days 00:02:35,2018-01-16 06:03:44,2018-01-16 06:04:17,Google Play Store
3,0,1,2018-01-16 06:25:54,Gmail,User Interaction,0 days 00:21:37,2018-01-16 06:25:54,2018-01-16 06:25:54,Calendar
4,0,1,2018-01-16 06:26:05,Google,Opened,0 days 00:00:11,2018-01-16 06:26:05,2018-01-16 06:26:10,Amazon Shopping
5,0,1,2018-01-16 06:26:10,Instagram,Opened,0 days 00:00:00,2018-01-16 06:26:10,2018-01-16 06:26:21,Google Chrome
...,...,...,...,...,...,...,...,...,...
96,0,23,2018-01-16 22:32:56,Gmail,Closed,0 days 00:00:01,2018-01-16 22:32:56,2018-01-16 22:32:56,Google
97,0,23,2018-01-16 22:32:57,Phone,Opened,0 days 00:00:01,2018-01-16 22:32:57,2018-01-16 22:34:07,Hangouts
98,0,23,2018-01-16 22:34:07,Gmail,Opened,0 days 00:00:00,2018-01-16 22:34:07,2018-01-16 22:34:47,Amazon Shopping


In [43]:
# create dicitonary user to accuracy
user_to_accuracy_dict = {}

# go through all interactions and check if app_name is in random_app_name for each user and save accuracy
for user_id, user_df in df_user:
    # iterate through all interactions for user_id
    accuracy = 0

    for index, row in user_df.iterrows():
        if row['app_name'] == row['random_app_name']:
            accuracy += 1

    user_to_accuracy_dict[user_id] = accuracy / len(user_df)

# print user to accuracy dictionary
user_to_accuracy_dict

{0: 0.060836501901140684,
 1: 0.06334841628959276,
 2: 0.06521739130434782,
 3: 0.08284023668639054,
 4: 0.13157894736842105,
 5: 0.03174045443897868,
 6: 0.04729082902546429,
 7: 0.043726235741444866,
 8: 0.13333333333333333,
 9: 0.046118721461187215,
 10: 0.125,
 11: 0.06609442060085836,
 12: 0.03162055335968379,
 13: 0.05153203342618384,
 14: 0.1702127659574468,
 15: 0.06201550387596899,
 16: 0.04830917874396135,
 17: 0.10493827160493827,
 18: 0.04808767613509282,
 19: 0.07471264367816093,
 20: 0.10975609756097561,
 21: 0.06164383561643835,
 22: 0.03260869565217391,
 23: 0.07058823529411765,
 24: 0.1717171717171717,
 25: 0.05437597099948213,
 26: 0.06127966356263142,
 27: 0.11194029850746269,
 28: 0.055299539170506916,
 29: 0.0914396887159533,
 30: 0.08053691275167785,
 31: 0.08647575843493054,
 32: 0.14942528735632185,
 33: 0.05409643277146217,
 34: 0.07954545454545454,
 35: 0.14285714285714285,
 36: 0.06561859193438141,
 37: 0.06218487394957983,
 38: 0.06769596199524941,
 39: 0.04

#### Results

In [44]:
# Calculate mean and standard deviation of accuracy
mean_accuracy = np.mean(list(user_to_accuracy_dict.values()))
std_accuracy = np.std(list(user_to_accuracy_dict.values()))

mean_accuracy, std_accuracy

(0.09545466117908077, 0.15162234054750554)

### Most opened app

292

### Most recently used